# Урок 22. Регрессия: предсказываем число

**Дата:** 18.02.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** урок 17 (регрессия как тип задачи), уроки 19–21 (train / test, baseline, сравнение моделей), урок 13 (графики).

До сих пор модели отвечали «да / нет». Но часто нужно **число**: сколько баллов наберёт ученик, сколько минут займёт дорога, сколько стоит квартира. Это **регрессия**. Сегодня обучим линейную регрессию, научимся измерять ошибку в понятных единицах и увидим, где модели нельзя верить.

## Цели урока

К концу занятия ты сможешь:

1. Обучать **линейную регрессию** и объяснять, что такое прямая `y = a · x + b`.
2. Измерять качество с помощью **MAE** — средней ошибки в единицах задачи.
3. Сравнивать модель с baseline «всегда среднее».
4. Строить график «факт — прогноз» и находить худшие ошибки.
5. Объяснять, почему модели опасно доверять **за пределами** данных.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Задача и данные |
| 5–15 | Ошибка в регрессии: MAE, baseline |
| 15–30 | Линейная регрессия: прямая и коэффициенты |
| 30–40 | Факт и прогноз, худшие ошибки, дерево для регрессии |
| 40–55 | Практика |
| 55–60 | Итог и домашнее задание |

## 1. Задача и данные

Задача: по числу часов занятий **предсказать балл** за итоговую работу (0–100), чтобы ученик понимал, чего ожидать. Цель — число, значит, это регрессия.

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def make_students(n, seed):
    """Сгенерировать учебную таблицу учеников (как на уроке 12)."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(200, seed=22)
X = df[["часы"]]
y = df["балл"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

`stratify` здесь не нужен: он работает только с классами, а у нас цель — число.

## 2. Как измерять ошибку в регрессии

Accuracy («доля угаданных») для чисел не подходит: предсказать балл **ровно** 73 почти невозможно, а промах на 1 балл и на 40 баллов — совсем разные вещи. Поэтому смотрят, **насколько** ошиблись.

**MAE** (*mean absolute error*, средняя абсолютная ошибка) — среднее отклонение прогноза от правды, без учёта знака:

$$\text{MAE} = \text{среднее}\;|\text{правда} - \text{прогноз}|$$

**Пример.** Правда: 70, 50, 90. Прогноз: 75, 40, 90. Ошибки без знака: 5, 10, 0. MAE = (5 + 10 + 0) / 3 = 5 баллов.

Главное достоинство MAE — она в **единицах задачи**: «модель ошибается в среднем на 6 баллов» понятно любому человеку.

### Baseline: всегда отвечать среднее

Самый простой прогноз для регрессии — **средний балл** по train для всех. Посчитаем его MAE вручную:

In [ ]:
mean_score = y_train.mean()
print("Средний балл по train:", round(mean_score, 1))

baseline_errors = (y_test - mean_score).abs()
print("MAE baseline (вручную):", round(baseline_errors.mean(), 1))

`.abs()` — **модуль** числа, как в математике: убирает минус (|−7| = 7). Применяется сразу ко всему столбцу.

То же в sklearn: `DummyRegressor` (по умолчанию отвечает средним) и функция `mean_absolute_error`.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

dummy = DummyRegressor()
dummy.fit(X_train, y_train)
print("MAE baseline:", round(mean_absolute_error(y_test, dummy.predict(X_test)), 1))

Если всем говорить «наберёшь 70», ошибка в среднем около 17 баллов. Любая полезная модель должна быть **заметно лучше**.

## 3. Линейная регрессия

Посмотрим на данные: балл явно растёт с часами, и облако точек похоже на **прямую**.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(X_train["часы"], y_train, alpha=0.5)
plt.title("Часы занятий и балл (train)")
plt.xlabel("Часов в неделю")
plt.ylabel("Балл")
plt.grid(alpha=0.3)
plt.show()

**Линейная регрессия** ищет прямую, которая проходит «ближе всего» ко всем точкам. Это знакомая из алгебры **линейная функция** `y = kx + b`, только вместо `y` — балл, а вместо `x` — часы:

$$\text{балл} = a \cdot \text{часы} + b$$

- **a** (коэффициент) — на сколько в среднем растёт балл, если добавить один час;
- **b** (свободный член) — прогноз при нуле часов.

Обучение — это подбор таких `a` и `b`, при которых точки в среднем ближе всего к прямой. «Ближе всего» здесь значит: если для каждой точки взять расстояние по вертикали до прямой, возвести в квадрат и сложить, сумма будет самой маленькой из возможных (поэтому способ называют **методом наименьших квадратов**). Схема в sklearn — та же: создать → `fit` → `predict`.

In [ ]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
lin.fit(X_train, y_train)

a = lin.coef_[0]
b = lin.intercept_
print(f"Модель: балл = {a:.2f} · часы + {b:.1f}")

pred = lin.predict(X_test)
print("MAE линейной регрессии:", round(mean_absolute_error(y_test, pred), 1))

**Как читать:** каждый дополнительный час занятий в неделю в среднем добавляет около 4.8 балла. Модель ошибается в среднем примерно на 6 баллов — почти в три раза лучше baseline.

`coef_` — список коэффициентов (по одному на признак), поэтому берём `[0]`; `intercept_` — свободный член.

Нарисуем прямую поверх точек:

In [ ]:
hours_line = pd.DataFrame({"часы": [1, 14]})
score_line = lin.predict(hours_line)

plt.figure(figsize=(6, 4))
plt.scatter(X_train["часы"], y_train, alpha=0.4, label="ученики (train)")
plt.plot(hours_line["часы"], score_line, color="crimson", linewidth=3, label="прямая модели")
plt.title("Линейная регрессия: балл от часов")
plt.xlabel("Часов в неделю")
plt.ylabel("Балл")
plt.legend()
plt.show()

## 4. Факт и прогноз

Для регрессии полезен график, где по горизонтали — **правда**, по вертикали — **прогноз**. У идеальной модели все точки лежали бы на диагонали.

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=0.6)
plt.plot([20, 100], [20, 100], "r--", label="идеальный прогноз")   # "r--" — красная пунктирная линия
plt.title("Правда и прогноз (test)")
plt.xlabel("Настоящий балл")
plt.ylabel("Прогноз модели")
plt.legend()
plt.show()

Посмотри на точки справа: у учеников с баллом 100 прогноз часто **больше 100**. Модель не знает, что шкала заканчивается на 100, — для прямой нет «потолка».

### Худшие ошибки

In [ ]:
table = X_test.copy()
table["правда"] = y_test
table["прогноз"] = pred.round(1)
table["ошибка"] = (table["правда"] - table["прогноз"]).abs()
table.sort_values("ошибка", ascending=False).head(5)

Худшие ошибки — это ученики, у которых балл сильно отличается от «типичного» для их часов: возможно, на них повлияло что-то, чего нет в признаках (болезнь, репетитор, волнение). Модель знает только то, что ей дали.

## 5. Модели нельзя верить за пределами данных

В обучающих данных часы — от 1 до 14. Что скажет модель про 30 часов в неделю?

In [ ]:
strange = pd.DataFrame({"часы": [0, 14, 20, 30]})
strange["прогноз"] = lin.predict(strange).round(1)
strange

Для 30 часов модель обещает почти **180 баллов** при шкале до 100! Прямая уходит вверх бесконечно, а реальность — нет. Предсказание за пределами диапазона обучающих данных называют **экстраполяцией**, и ему нельзя доверять: модель там «ни разу не была».

**Правило:** записывай в ограничения модели диапазон данных, на которых она обучалась.

## 6. Несколько признаков и дерево для регрессии

Линейная регрессия умеет работать с несколькими признаками: у каждого будет свой коэффициент, а формула становится длиннее: `балл = a₁ · часы + a₂ · посещаемость + a₃ · сон + b`.

In [ ]:
X2 = df[["часы", "посещаемость", "сон"]]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.25, random_state=42)

lin2 = LinearRegression()
lin2.fit(X2_train, y2_train)
print(pd.Series(lin2.coef_, index=X2.columns).round(2))
print("MAE:", round(mean_absolute_error(y2_test, lin2.predict(X2_test)), 1))

Коэффициент у сна маленький — модель «поняла», что сон почти не влияет (как мы и заложили). А вот MAE не улучшилась, а даже стала чуть хуже: часы и посещаемость сильно связаны между собой (урок 12), и второй признак почти не добавляет новой информации.

Дерево решений тоже умеет предсказывать числа — `DecisionTreeRegressor`. В каждом «листе» дерева оно отвечает **средним** баллом учеников, попавших туда.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

for depth in [3, 10]:
    t = DecisionTreeRegressor(max_depth=depth, random_state=0)
    t.fit(X2_train, y2_train)
    mae_train = mean_absolute_error(y2_train, t.predict(X2_train))
    mae_test = mean_absolute_error(y2_test, t.predict(X2_test))
    print(f"дерево глубины {depth:2}: MAE train {mae_train:.1f}, test {mae_test:.1f}")

Глубокое дерево почти без ошибок на train и заметно хуже на test — снова переобучение. На этих данных линейная регрессия честнее и проще.

## 7. Практика

### Практика 1. Посещаемость вместо часов (8 минут)

Обучи линейную регрессию **только на посещаемости**. Выведи формулу модели и MAE на test. Что предсказывает лучше — часы или посещаемость?

In [ ]:
# Практика 1

<details>
<summary>Решение</summary>

```python
Xa = df[["посещаемость"]]
Xa_train, Xa_test, ya_train, ya_test = train_test_split(Xa, y, test_size=0.25, random_state=42)
lin_a = LinearRegression()
lin_a.fit(Xa_train, ya_train)
print(f"балл = {lin_a.coef_[0]:.2f} · посещаемость + ({lin_a.intercept_:.1f})")
print("MAE:", round(mean_absolute_error(ya_test, lin_a.predict(Xa_test)), 1))
```
Посещаемость предсказывает хуже часов: связь с баллом у неё слабее (вспомни корреляции на уроке 12).
</details>

### Практика 2. Прогноз для себя (5 минут)

Сколько баллов модель `lin` предсказывает при 4, 8 и 12 часах в неделю? Проверь по формуле `a · часы + b` вручную для одного значения.

In [ ]:
# Практика 2

<details>
<summary>Решение</summary>

```python
new = pd.DataFrame({"часы": [4, 8, 12]})
print(lin.predict(new).round(1))
print("Вручную для 8 часов:", round(a * 8 + b, 1))
```
</details>

### Практика 3. Ограничения модели (5 минут)

Напиши 3 ограничения модели `lin`: где ей нельзя доверять и почему. Подсказки: диапазон часов, шкала 0–100, чего не знает модель.

1. …
2. …
3. …

## Работа с ИИ-помощником

- «Объясни, что означает коэффициент 4.8 в линейной регрессии балла от часов.»
- «Почему MAE удобнее объяснять людям, чем другие метрики?»
- «Моя модель предсказывает больше 100 баллов. Почему так и что с этим делать?»

## Мини-глоссарий урока 22

| Термин | Смысл |
|--------|-------|
| **Регрессия** | предсказание числа |
| **Линейная регрессия** | модель-прямая: `y = a · x + b` |
| **Коэффициент** | на сколько меняется прогноз при росте признака на 1 |
| **Свободный член** | прогноз, когда все признаки равны 0 |
| **MAE** | средняя абсолютная ошибка в единицах задачи |
| **Baseline для регрессии** | всегда предсказывать среднее по train |
| **Экстраполяция** | прогноз за пределами диапазона обучающих данных |

## Итог урока

1. В регрессии ошибку меряют **в единицах задачи** — MAE.
2. Любую модель сравнивают с baseline «всегда среднее».
3. Модель знает только диапазон своих данных: за его пределами прогнозы могут быть абсурдными.

## Домашнее задание (20 минут)

Поставь задачу регрессии **на других данных**: сгенерируй `make_students(200, seed=5)` и предскажи балл по часам.

1. Найди MAE baseline и линейной регрессии на test.
2. Выведи формулу модели. Похож ли коэффициент на сегодняшний (≈ 4.8)?
3. Напиши 2 предложения: что модель умеет и где её нельзя применять.

In [ ]:
# Домашнее задание

## Самопроверка

1. Почему для регрессии не подходит accuracy?
2. Что показывает MAE = 6?
3. Что означают `a` и `b` в формуле линейной регрессии?
4. Какой baseline у задачи регрессии?
5. Почему нельзя доверять прогнозу для 30 часов?

**Следующий урок (25.02.2027):** переобучение и честное сравнение моделей.